In [33]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_classes=2,
    n_informative=8,
    n_redundant=2,
    n_repeated=0,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
model = LogisticRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.73      0.65      0.69       130
           1       0.66      0.74      0.70       120

    accuracy                           0.70       250
   macro avg       0.70      0.70      0.70       250
weighted avg       0.70      0.70      0.70       250



# Implement K-Fold

In [35]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)


model = LogisticRegression()

# for train_index, test_index in kf.split([12,14,15,15,21]):
#     print(train_index, test_index)

scores = []

for train_index, test_index in kf.split(X,y):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    scores.append(score)

print(scores)

[0.675, 0.715, 0.72, 0.645, 0.72]


## Evaluate Logistic Regression

In [45]:
import numpy as np
from sklearn.model_selection import cross_val_score
lg_scores = cross_val_score(LogisticRegression(), X, y, cv=kf)
print(lg_scores)
print(np.average(lg_scores))

[0.675 0.715 0.72  0.645 0.72 ]
0.6950000000000001


## Evaluate Decision Tree

In [50]:
from sklearn.tree import DecisionTreeClassifier
dt_scores = cross_val_score(DecisionTreeClassifier(criterion="entropy"), X, y, cv=kf)
print(dt_scores)
print(np.average(dt_scores))

[0.775 0.805 0.845 0.76  0.825]
0.8019999999999999


## Evaluate with Random Forest Classifier

In [47]:
from sklearn.ensemble import RandomForestClassifier
rf_scores = cross_val_score(RandomForestClassifier(n_estimators=20), X, y, cv=kf)
print(rf_scores)
print(np.average(rf_scores))

[0.87  0.875 0.865 0.87  0.855]
0.8670000000000002


## Evaluate with XGBoost

In [48]:
from xgboost import XGBClassifier
xg_scores = cross_val_score(XGBClassifier(), X, y, cv=kf)
print(xg_scores)
print(np.average(xg_scores))

[0.88  0.905 0.88  0.895 0.925]
0.897


## Evaluate with XGBoost with scoring = "roc_auc"


In [51]:
from xgboost import XGBClassifier
xg_scores = cross_val_score(XGBClassifier(), X, y, cv=kf, scoring='roc_auc') #by default the scoring is accuracy
print(xg_scores)
print(np.average(xg_scores))

[0.93807708 0.95338135 0.95338135 0.96053686 0.97661582]
0.9563984916952606


In [53]:
from sklearn.model_selection import cross_validate

cross_validate(LogisticRegression(), X, y, cv=kf, scoring=["accuracy",'roc_auc'])


{'fit_time': array([0.01102901, 0.0013268 , 0.00093412, 0.00092483, 0.00087595]),
 'score_time': array([0.0020802 , 0.0012579 , 0.00090194, 0.00081992, 0.00097489]),
 'test_accuracy': array([0.675, 0.715, 0.72 , 0.645, 0.72 ]),
 'test_roc_auc': array([0.7393617 , 0.77681072, 0.79251701, 0.74919872, 0.77569249])}